# Pandas for Data Wrangling (Advanced) 🚢

### ISA 383: Python for Business Analytics

This is the second of two notebooks on pandas.
The Basic notebook taught you to read, inspect, reshape, select, filter, and sort a single table.
Here you will **combine** tables with `concat()`, `merge()`, and `join()`, and **summarize** them with `groupby()`, aggregations, and pivot tables.
We continue the investigation of the RMS *Titanic* passenger list.

## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.

## Learning objectives 🎯

By the end of this notebook, you should be able to:

1. Stack tables with `concat()` and explain when stacking is the right operation.
2. Combine related tables with `merge()` and `join()`, choose the correct join type, and validate key relationships.
3. Summarize groups with `groupby()` using `size()`, `count()`, `mean()`, `agg()`, and named aggregations.
4. Use `transform()` and `filter()` for group-aware calculations at the row level.
5. Build pivot tables and crosstabs, and bin numeric variables with `cut()` and `qcut()` to compare groups.

## 1. Where we left off 🧊

In April 1912, the *Titanic* sank on its maiden voyage after striking an iceberg, with lifeboat space for only about half of the 2,224 people aboard.
Two official inquiries asked who had been saved and why.
In the Basic notebook, our filters showed that 38.4% of the 891 passengers in our file survived, that women survived far more often than men, and that first-class passengers survived far more often than third-class passengers.

Those comparisons looked at one variable at a time.
Real questions usually involve several variables together, and the information we need is often spread over several tables.
Your remaining inquiry questions:

| # | Inquiry question | pandas skill |
|---|---|---|
| Q6 | Did first class buy a better chance, and did that depend on sex and age? | `groupby()` with two keys, `pivot_table()`, `cut()` |
| Q7 | Did the port of embarkation matter, or was it really ticket class? | `merge()`, `crosstab()` |
| Q8 | How much did each passenger really pay when several people shared a ticket? | `groupby()` + `transform()` |
| Q9 | Did travelling with family help? | `cut()`, named aggregation |

Look for the 🕵️ symbol in the cells that answer them.

## 2. Setup 🧰

We download the course data from the course GitHub repository, load the passenger file and the two extra sheets from the Excel workbook, and recreate the `family_size` column from the Basic notebook.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

from urllib.request import urlretrieve

DATA_DIR = Path("/content/isa383/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download the course data from the course GitHub repository (only the first time).
DATA_URL = "https://raw.githubusercontent.com/harshvardhaniimi/isa383-google-colab/main/data/"
for filename in ["titanic.csv", "titanic_extra.xlsx"]:
    if not (DATA_DIR / filename).exists():
        urlretrieve(DATA_URL + filename, DATA_DIR / filename)
print("Course data is ready.")

titanic = pd.read_csv(DATA_DIR / "titanic.csv")
ports = pd.read_excel(DATA_DIR / "titanic_extra.xlsx", sheet_name="ports")
board_of_trade = pd.read_excel(DATA_DIR / "titanic_extra.xlsx", sheet_name="board_of_trade")

titanic["family_size"] = titanic["SibSp"] + titanic["Parch"] + 1

print("titanic:", titanic.shape, "| ports:", ports.shape, "| board_of_trade:", board_of_trade.shape)

Course data is ready.
titanic: (891, 13) | ports: (4, 6) | board_of_trade: (4, 9)


## 3. Stacking tables with `concat()` 🧱

`pd.concat()` glues tables together along an axis.
With `axis=0` (the default), it stacks rows: use it when several tables contain **more rows of the same kind**, such as monthly sales files with identical columns.
With `axis=1`, it places tables side by side and lines them up by their index labels.

`concat()` does not look at the *values* in any column; it only matches column names (when stacking) or index labels (when placing side by side).
To match rows by a key such as a ticket number or a port code, you need `merge()`, covered next.

### 🧭 Syntax first: Stacking with concat

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
pd.concat(
    [df1, df2, df3],
    axis=0,               # 0 = stack rows, 1 = place side by side
    ignore_index=True,    # renumber rows 0, 1, 2, ...
    keys=["a", "b", "c"], # label which table each row came from
    join="outer",         # "outer" keeps all columns, "inner" keeps shared columns only
)
```

Documentation: [concat](https://pandas.pydata.org/docs/reference/api/pandas.concat.html)

Suppose the inquiry's clerks delivered the passenger list as three separate files, one per ticket class.
We simulate that by splitting our table, then put it back together.

In [2]:
first = titanic[titanic["Pclass"] == 1]
second = titanic[titanic["Pclass"] == 2]
third = titanic[titanic["Pclass"] == 3]

print("Rows per file:", len(first), len(second), len(third))

Rows per file: 216 184 491


In [3]:
manifest = pd.concat([first, second, third], ignore_index=True)

print("Rows after concat:", len(manifest))
manifest[["Name", "Pclass"]].head()

Rows after concat: 891


,Name,Pclass
0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1
1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1
2,"McCarthy, Mr. Timothy J",1
3,"Bonnell, Miss. Elizabeth",1
4,"Sloper, Mr. William Thompson",1


In [4]:
# keys= records which file each row came from; reset_index() turns that label into a column.
manifest_labeled = (
    pd.concat([first, second, third], keys=["first.csv", "second.csv", "third.csv"])
    .reset_index(level=0)
    .rename(columns={"level_0": "source_file"})
)
manifest_labeled["source_file"].value_counts()

source_file
third.csv     491
first.csv     216
second.csv    184
Name: count, dtype: int64

In [5]:
# Suppose the third-class clerk did not record cabins at all.
third_without_cabin = third.drop(columns=["Cabin"])

outer = pd.concat([first, third_without_cabin], ignore_index=True)                # default join="outer"
inner = pd.concat([first, third_without_cabin], ignore_index=True, join="inner")

print("outer:", outer.shape, "-> missing cabins filled with NaN:", outer["Cabin"].isna().sum())
print("inner:", inner.shape, "-> Cabin column kept?", "Cabin" in inner.columns)

outer: (707, 13) -> missing cabins filled with NaN: 531
inner: (707, 12) -> Cabin column kept? False


In [6]:
# axis=1 places Series side by side, aligned on their index labels (here, the class number).
passengers_by_class = titanic["Pclass"].value_counts()
survivors_by_class = titanic.loc[titanic["Survived"] == 1, "Pclass"].value_counts()

class_table = pd.concat([passengers_by_class, survivors_by_class], axis=1, keys=["passengers", "survivors"])
class_table["survival_rate"] = (class_table["survivors"] / class_table["passengers"]).round(3)
class_table.sort_index()

,passengers,survivors,survival_rate
Pclass,,,
1,216,136,0.630
2,184,87,0.473
3,491,119,0.242


The two Series were in different orders, yet each class lined up with its own counts because `concat(axis=1)` matches index labels.

> 💭 **Quick thought question:** Two files describe the same thing but one calls the column `Fare` and the other `fare_gbp`.
What happens if you stack them with `concat()`, and how would you fix it first?

### 🧩 Guided practice: Split by port, then stack

Split `titanic` into three DataFrames by `Embarked` (`"S"`, `"C"`, `"Q"`).
Stack them back with `concat()` using `keys=` so you can tell the ports apart.
How many rows does the result have, and why is it not 891?

In [7]:
# TODO: Create the three port DataFrames.

# TODO: Stack them with keys and check the number of rows.

## 4. Combining related tables with `merge()` 🔗

`merge()` matches rows from two tables using the values in one or more **key** columns, like a SQL `JOIN` or Excel's `XLOOKUP`.
Before merging, ask two questions.

**What is the relationship between the keys?**
Many passengers boarded at one port, so passengers-to-ports is **many-to-one**.
One-to-one and many-to-many relationships also exist; many-to-many is usually a warning sign.

**Which rows should survive the merge?**

| `how=` | Keeps | Typical use |
|---|---|---|
| `"inner"` | only keys found in **both** tables | analysis where unmatched rows are useless |
| `"left"` | every row of the **left** table | enriching a main table with a lookup (the most common case) |
| `"right"` | every row of the **right** table | rarely needed; swap the tables and use `"left"` |
| `"outer"` | every row from **both** tables | auditing: finding what did not match |

### 🧭 Syntax first: Merging tables

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
left.merge(
    right,
    how="left",                         # "inner", "left", "right", or "outer"
    on="key",                           # when the key has the same name in both tables
    left_on="key_a", right_on="key_b",  # when the names differ
    suffixes=("_x", "_y"),              # added to other columns that share a name
    validate="many_to_one",             # raise an error if the relationship is wrong
    indicator=True,                     # add a _merge column: both / left_only / right_only
)
```

Documentation: [DataFrame.merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html) · [Merge guide](https://pandas.pydata.org/docs/user_guide/merging.html)

In [8]:
# The lookup table: one row per port the ship left from.
ports

,code,port,country_today,departure_date,stop_number,note
0,B,Belfast,United Kingdom,1912-04-02,0,"Shipyard; delivery trip only, no passengers in..."
1,S,Southampton,United Kingdom,1912-04-10,1,Maiden voyage begins
2,C,Cherbourg,France,1912-04-10,2,Evening call in France
3,Q,Queenstown,Ireland,1912-04-11,3,Last stop before the Atlantic crossing (now ca...


In [9]:
with_ports = titanic.merge(
    ports[["code", "port", "departure_date", "stop_number"]],
    how="left",
    left_on="Embarked",      # the key is called Embarked in titanic ...
    right_on="code",         # ... and code in ports
    validate="many_to_one",  # each port code must appear only once in ports
    indicator=True,
)

with_ports[["Name", "Embarked", "code", "port", "departure_date", "_merge"]].head()

,Name,Embarked,code,port,departure_date,_merge
0,"Braund, Mr. Owen Harris",S,S,Southampton,1912-04-10,both
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C,C,Cherbourg,1912-04-10,both
2,"Heikkinen, Miss. Laina",S,S,Southampton,1912-04-10,both
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",S,S,Southampton,1912-04-10,both
4,"Allen, Mr. William Henry",S,S,Southampton,1912-04-10,both


In [10]:
with_ports["_merge"].value_counts()

_merge
both          889
left_only       2
right_only      0
Name: count, dtype: int64

In [11]:
# Which passengers found no matching port?
with_ports.loc[with_ports["_merge"] == "left_only", ["Name", "Ticket", "Cabin", "Embarked", "port"]]

,Name,Ticket,Cabin,Embarked,port
61,"Icard, Miss. Amelie",113572,B28,NaN,NaN
829,"Stone, Mrs. George Nelson (Martha Evelyn)",113572,B28,NaN,NaN


Miss Amelie Icard and Mrs. George Nelson Stone shared ticket 113572 and cabin B28, but their port of embarkation is missing.
A left merge kept them, with `NaN` for every port column, and `indicator=True` made them easy to find.
*Encyclopedia Titanica* records that both boarded at Southampton; we will use that kind of evidence when we fill missing values in the cleaning notebook.

In [12]:
# How many rows does each join type keep? A loop makes the comparison easy.
for how in ["inner", "left", "right", "outer"]:
    result = titanic.merge(ports, how=how, left_on="Embarked", right_on="code")
    print(f"{how:>5}: {len(result):>4} rows")

inner:  889 rows
 left:  891 rows
right:  890 rows
outer:  892 rows


In [13]:
# An outer merge with an indicator is an audit: it shows what did not match on either side.
audit = titanic.merge(ports, how="outer", left_on="Embarked", right_on="code", indicator=True)
audit.loc[audit["_merge"] != "both", ["Name", "Embarked", "code", "port", "_merge"]]

,Name,Embarked,code,port,_merge
0,NaN,NaN,B,Belfast,right_only
890,"Icard, Miss. Amelie",NaN,NaN,NaN,left_only
891,"Stone, Mrs. George Nelson (Martha Evelyn)",NaN,NaN,NaN,left_only


- `inner` dropped the two passengers with no port (889 rows).
- `left` kept all 891 passengers.
- `right` kept every port, so Belfast appears once with no passenger attached (889 + 1 = 890 rows).
- `outer` kept everything from both sides (892 rows).

### When keys repeat on both sides: many-to-many ⚠️

If a key value appears several times in **both** tables, `merge()` pairs every match with every other match.
That is sometimes exactly what you want, for example to list who travelled with whom.
It is also the most common way an analysis silently gains extra rows.

In [14]:
# Who travelled with whom? Merge the passenger list with itself on Ticket.
people_on_tickets = titanic[["Ticket", "Name", "Age"]]

companions = people_on_tickets.merge(
    people_on_tickets,
    on="Ticket",
    suffixes=("", "_companion"),   # both sides have Name and Age, so label the right-hand copies
)
companions = companions[companions["Name"] != companions["Name_companion"]]

print("Companion pairs:", len(companions))
companions[companions["Ticket"] == "PC 17755"]

Companion pairs: 702


,Ticket,Name,Age,Name_companion,Age_companion
464,PC 17755,"Ward, Miss. Anna",35.0,"Cardeza, Mr. Thomas Drake Martinez",36.0
465,PC 17755,"Ward, Miss. Anna",35.0,"Lesurer, Mr. Gustave J",35.0
1202,PC 17755,"Cardeza, Mr. Thomas Drake Martinez",36.0,"Ward, Miss. Anna",35.0
1204,PC 17755,"Cardeza, Mr. Thomas Drake Martinez",36.0,"Lesurer, Mr. Gustave J",35.0
1314,PC 17755,"Lesurer, Mr. Gustave J",35.0,"Ward, Miss. Anna",35.0
1315,PC 17755,"Lesurer, Mr. Gustave J",35.0,"Cardeza, Mr. Thomas Drake Martinez",36.0


In [15]:
# validate= protects you: Ticket is NOT unique in the right table, so many_to_one must fail.
try:
    titanic.merge(people_on_tickets, on="Ticket", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


> 💭 **Quick thought question:** After a left merge, the result has more rows than the left table.
What does that tell you about the key in the right table?

### 🧩 Guided practice: Inner merge

Merge `titanic` with `ports` using `how="inner"`.
How many rows does the result have?
Use `value_counts()` on the `port` column to count passengers per port, and explain in one sentence why two passengers are missing.

In [16]:
# TODO: Inner merge, then count passengers per port.

## 5. Joining on the index with `join()` 🪢

`DataFrame.join()` is a shortcut for merging when the right-hand table is **indexed by the key**.
It performs a left join by default.
It is convenient for lookup tables that already use the key as their index and for attaching group summaries.

### 🧭 Syntax first: Joining on the index

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
left.join(right, on="key_column", how="left")       # right is indexed by the key
left.join(right, how="left", rsuffix="_right")      # both tables share the same index
```

Documentation: [DataFrame.join](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html)

In [17]:
class_names = pd.DataFrame(
    {"class_name": ["First", "Second", "Third"]},
    index=pd.Index([1, 2, 3], name="Pclass"),
)
class_names

,class_name
Pclass,
1,First
2,Second
3,Third


In [18]:
# Match titanic's Pclass column against class_names' index.
titanic_named = titanic.join(class_names, on="Pclass")
titanic_named[["Name", "Pclass", "class_name"]].head()

,Name,Pclass,class_name
0,"Braund, Mr. Owen Harris",3,Third
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,First
2,"Heikkinen, Miss. Laina",3,Third
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,First
4,"Allen, Mr. William Henry",3,Third


In [19]:
# The same idea with the ports table, after making the port code its index.
ports_by_code = ports.set_index("code")
titanic.join(ports_by_code[["port"]], on="Embarked")[["Name", "Embarked", "port"]].head()

,Name,Embarked,port
0,"Braund, Mr. Owen Harris",S,Southampton
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C,Cherbourg
2,"Heikkinen, Miss. Laina",S,Southampton
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",S,Southampton
4,"Allen, Mr. William Henry",S,Southampton


| Tool | Matches rows by | Use it when |
|---|---|---|
| `pd.concat()` | position (rows) or index labels (columns) | you have more rows, or more columns, of the same kind |
| `df.merge()` | values in key **columns** | two related tables share a key column |
| `df.join()` | the right table's **index** | the lookup table is already indexed by the key |

> 💭 **Quick thought question:** Anything `join()` can do, `merge()` can also do.
Why might a team still prefer one of them as a house style?

## 6. `groupby()`: split, apply, combine 🧮

Most business questions compare groups: revenue by region, churn by plan, survival by class.
`groupby()` follows a three-step pattern known as **split-apply-combine**.

```text
                 split by Pclass             apply mean()          combine
891 rows  ──┬──> Pclass 1 (216 rows) ──────> 0.630 ──┐
            ├──> Pclass 2 (184 rows) ──────> 0.473 ──┼──> one row per class
            └──> Pclass 3 (491 rows) ──────> 0.242 ──┘
```

The result has one row per group.
With two grouping keys, the result has one row per *combination* of keys.

### 🧭 Syntax first: Grouping

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.groupby("key")["value"].mean()
df.groupby(["key_a", "key_b"])["value"].mean()
df.groupby("key", as_index=False)["value"].mean()   # keys stay as ordinary columns

df.groupby("key").size()             # rows per group
df.groupby("key")["value"].count()   # NON-MISSING values per group
series_with_two_keys.unstack()       # move the inner key into the columns
```

Documentation: [groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) · [GroupBy guide](https://pandas.pydata.org/docs/user_guide/groupby.html) · [unstack](https://pandas.pydata.org/docs/reference/api/pandas.Series.unstack.html)

In [20]:
# 🕵️ Q6 (part 1): Did first class buy a better chance?
titanic.groupby("Pclass")["Survived"].mean().round(3)

Pclass
1    0.630
2    0.473
3    0.242
Name: Survived, dtype: float64

In [21]:
# size() counts rows; count() counts non-missing values in one column.
by_class = titanic.groupby("Pclass")

pd.DataFrame(
    {
        "passengers": by_class.size(),
        "ages_recorded": by_class["Age"].count(),
        "survivors": by_class["Survived"].sum(),
    }
)

,passengers,ages_recorded,survivors
Pclass,,,
1,216,186,136
2,184,173,87
3,491,355,119


In [22]:
# Two keys: one row per (class, sex) combination.
rate_by_class_sex = titanic.groupby(["Pclass", "Sex"])["Survived"].mean().round(3)
rate_by_class_sex

Pclass  Sex   
1       female    0.968
        male      0.369
2       female    0.921
        male      0.157
3       female    0.500
        male      0.135
Name: Survived, dtype: float64

In [23]:
# unstack() turns the long result into a wide table that is easier to read.
rate_by_class_sex.unstack()

Sex,female,male
Pclass,,
1,0.968,0.369
2,0.921,0.157
3,0.500,0.135


This small table carries the main story of the disaster.
Within every class, women survived far more often than men.
Within each sex, survival fell with class: 96.8% of first-class women survived, but only 50.0% of third-class women; 36.9% of first-class men survived, but only 13.5% of third-class men.
Sex mattered most, and class mattered within each sex.

In [24]:
# as_index=False keeps the keys as ordinary columns, which is convenient for merging and plotting.
titanic.groupby(["Pclass", "Sex"], as_index=False)["Survived"].mean()

,Pclass,Sex,Survived
0,1,female,0.968085
1,1,male,0.368852
2,2,female,0.921053
3,2,male,0.157407
4,3,female,0.500000
5,3,male,0.135447


In [25]:
# A GroupBy object can be looped over: each item is (group key, sub-DataFrame).
for port_code, group in titanic.groupby("Embarked"):
    print(f"{port_code}: {len(group):>3} passengers, survival rate {group['Survived'].mean():.1%}")

C: 168 passengers, survival rate 55.4%
Q:  77 passengers, survival rate 39.0%
S: 644 passengers, survival rate 33.7%


In [26]:
# 🕵️ Q7 (part 1): Did the port of embarkation matter?
with_ports.groupby("port")["Survived"].mean().round(3).sort_values(ascending=False)

port
Cherbourg      0.554
Queenstown     0.390
Southampton    0.337
Name: Survived, dtype: float64

Passengers who boarded at Cherbourg survived far more often than those from Southampton.
Did boarding in France protect people, or were different kinds of people boarding there?
We return to this in Section 8.

> 💭 **Quick thought question:** Why is `titanic.groupby("Pclass").size()` different from `titanic.groupby("Pclass")["Age"].count()`?
Which one should you use as the denominator of a survival rate?

### 🧩 Guided practice: Sex and port

Compute the survival rate for every combination of `Sex` and `Embarked`, then `unstack()` the result so that ports are columns.
Does the Cherbourg advantage hold for both women and men?

In [27]:
# TODO: Group by Sex and Embarked, compute the mean of Survived, then unstack.

## 7. Aggregations: `agg()`, `transform()`, and `filter()` 📊

One number per group is often not enough.
`agg()` computes several summaries at once, and **named aggregation** gives each output column a readable name.
`transform()` is different: it returns one value **per original row**, so you can compare each passenger with their group.
`filter()` keeps or drops **whole groups** based on a group-level rule.

### 🧭 Syntax first: Aggregating, transforming, and filtering groups

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.groupby("key")["value"].agg(["mean", "median", "max"])
df.groupby("key").agg({"value_a": "mean", "value_b": ["min", "max"]})

df.groupby("key").agg(
    output_name=("column", "function"),          # built-in name such as "mean"
    other_output=("column", custom_function),    # or your own function
)

df.groupby("key")["value"].transform("mean")     # same length as df
df.groupby("key").filter(lambda group: len(group) >= 4)
```

Documentation: [agg](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html) · [Named aggregation](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation) · [transform](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.transform.html) · [filter](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.filter.html)

In [28]:
# A list of functions applied to one column.
titanic.groupby("Pclass")["Fare"].agg(["mean", "median", "min", "max"]).round(2)

,mean,median,min,max
Pclass,,,,
1,84.15,60.29,0.0,512.33
2,20.66,14.25,0.0,73.50
3,13.68,8.05,0.0,69.55


In every class the mean fare is well above the median, a sign of a few very expensive tickets pulling the average up (a right-skewed distribution).
For skewed money values, the median is usually the more honest "typical" value.

In [29]:
# A dictionary applies different functions to different columns.
titanic.groupby("Pclass").agg({"Age": ["mean", "median"], "Fare": "median", "Survived": "mean"}).round(2)

Age          Fare Survived
         mean median median     mean
Pclass                              
1       38.23   37.0  60.29     0.63
2       29.88   29.0  14.25     0.47
3       25.14   24.0   8.05     0.24

In [30]:
# Named aggregation: output_name=(input_column, function). This is the most readable form.
class_report = (
    titanic.groupby("Pclass")
    .agg(
        passengers=("PassengerId", "size"),
        survivors=("Survived", "sum"),
        survival_rate=("Survived", "mean"),
        median_age=("Age", "median"),
        median_fare=("Fare", "median"),
        women=("Sex", lambda values: (values == "female").sum()),
    )
    .round(3)
)
class_report

,passengers,survivors,survival_rate,median_age,median_fare,women
Pclass,,,,,,
1,216,136,0.630,37.0,60.287,94
2,184,87,0.473,29.0,14.250,76
3,491,119,0.242,24.0,8.050,144


In [31]:
# Any function that takes a Series and returns one value can be an aggregation.
def share_missing(values):
    # Share of values in this group that are missing.
    return values.isna().mean()

titanic.groupby("Pclass").agg(
    missing_age=("Age", share_missing),
    missing_cabin=("Cabin", share_missing),
).round(3)

,missing_age,missing_cabin
Pclass,,
1,0.139,0.185
2,0.060,0.913
3,0.277,0.976


Cabin numbers are missing for 98% of third-class passengers but only 19% of first-class passengers.
Missing values are not scattered at random: *where* they are missing carries information, a point the cleaning notebook builds on.

In [32]:
# 🕵️ Q8: How much did each passenger really pay?
# transform("size") gives every row the number of passengers on its ticket.
titanic["ticket_group_size"] = titanic.groupby("Ticket")["PassengerId"].transform("size")
titanic["fare_per_person"] = (titanic["Fare"] / titanic["ticket_group_size"]).round(2)

titanic.loc[
    titanic["Ticket"].isin(["PC 17755", "CA. 2343"]),
    ["Name", "Ticket", "Fare", "ticket_group_size", "fare_per_person"],
]

,Name,Ticket,Fare,ticket_group_size,fare_per_person
159,"Sage, Master. Thomas Henry",CA. 2343,69.5500,7,9.94
180,"Sage, Miss. Constance Gladys",CA. 2343,69.5500,7,9.94
201,"Sage, Mr. Frederick",CA. 2343,69.5500,7,9.94
258,"Ward, Miss. Anna",PC 17755,512.3292,3,170.78
324,"Sage, Mr. George John Jr",CA. 2343,69.5500,7,9.94
679,"Cardeza, Mr. Thomas Drake Martinez",PC 17755,512.3292,3,170.78
737,"Lesurer, Mr. Gustave J",PC 17755,512.3292,3,170.78
792,"Sage, Miss. Stella Anna",CA. 2343,69.5500,7,9.94
846,"Sage, Mr. Douglas Bullen",CA. 2343,69.5500,7,9.94
863,"Sage, Miss. Dorothy Edith ""Dolly""",CA. 2343,69.5500,7,9.94


The Cardeza party's £512.33 works out to £170.78 per person among the three people in our file, and the Sage family's £69.55 ticket to £9.94 each.
Our file is only a subset, though: Charlotte Cardeza and four more members of the Sage family are missing from it.
Because `ticket_group_size` can only count companions who appear in our data, `fare_per_person` is an **upper bound** on what each person really paid.

In [33]:
titanic.groupby("Pclass")[["Fare", "fare_per_person"]].median().round(2)

,Fare,fare_per_person
Pclass,,
1,60.29,35.25
2,14.25,13.00
3,8.05,7.85


In [34]:
# transform() lets each row be compared with its own group.
titanic["class_median_fare"] = titanic.groupby("Pclass")["fare_per_person"].transform("median")
titanic["paid_vs_class"] = (titanic["fare_per_person"] / titanic["class_median_fare"]).round(2)

(
    titanic.sort_values("paid_vs_class", ascending=False)
    [["Name", "Pclass", "fare_per_person", "class_median_fare", "paid_vs_class"]]
    .head()
)

,Name,Pclass,fare_per_person,class_median_fare,paid_vs_class
527,"Farthing, Mr. John",1,221.78,35.25,6.29
377,"Widener, Mr. Harry Elkins",1,211.50,35.25,6.00
258,"Ward, Miss. Anna",1,170.78,35.25,4.84
737,"Lesurer, Mr. Gustave J",1,170.78,35.25,4.84
679,"Cardeza, Mr. Thomas Drake Martinez",1,170.78,35.25,4.84


In [35]:
# filter() keeps every row of the groups that pass a test: tickets shared by four or more people.
large_parties = titanic.groupby("Ticket").filter(lambda group: len(group) >= 4)

print("Passengers on tickets shared by 4+ people:", len(large_parties))
print("Their survival rate:", round(large_parties["Survived"].mean(), 3))

Passengers on tickets shared by 4+ people: 93
Their survival rate: 0.29


> 💭 **Quick thought question:** `agg()` returned one row per class, while `transform()` returned one value for each of the 891 passengers.
Which would you use to add a "class average fare" column to the passenger table, and which to build a summary slide?

### 🧩 Guided practice: A port report

Use named aggregation to build a table with one row per `Embarked` value and these columns: `passengers`, `survival_rate`, `median_fare_per_person`, and `share_first_class`.
*Hint:* for the last one, use `("Pclass", lambda values: (values == 1).mean())`.

In [36]:
# TODO: Group by Embarked and use named aggregation.

## 8. Pivot tables, crosstabs, and binning 🧾

A **pivot table** is `groupby()` plus `unstack()` in one step, with optional row and column totals (`margins=True`).
Unlike `pivot()` from the Basic notebook, `pivot_table()` *aggregates* when several rows share the same index and column, so duplicates are not a problem.
`pd.crosstab()` counts how often combinations occur, and `normalize=` turns those counts into shares.

To compare groups along a numeric variable such as age, first **bin** it.
`pd.cut()` uses bin edges you choose, while `pd.qcut()` creates groups with (roughly) equal numbers of rows, such as quartiles.

### 🧭 Syntax first: Pivot tables, crosstabs, and bins

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.pivot_table(
    values="value", index="row_key", columns="column_key",
    aggfunc="mean", margins=True, observed=True,
)
pd.crosstab(df["row_key"], df["column_key"], normalize="index")   # shares within each row

pd.cut(df["number"], bins=[0, 13, 18, 100], labels=["child", "teen", "adult"], right=False)
pd.qcut(df["number"], q=4, labels=["Q1", "Q2", "Q3", "Q4"])
```

Documentation: [pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) · [crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) · [cut](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) · [qcut](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html)

In [37]:
# The class x sex survival table in one call, with totals in the "All" row and column.
titanic.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="mean", margins=True).round(3)

Sex,female,male,All
Pclass,,,
1,0.968,0.369,0.630
2,0.921,0.157,0.473
3,0.500,0.135,0.242
All,0.742,0.189,0.384


In [38]:
# Bin ages into labeled groups. right=False makes each bin include its left edge: [0, 13), [13, 18), ...
titanic["age_group"] = pd.cut(
    titanic["Age"],
    bins=[0, 13, 18, 40, 60, 81],
    labels=["child (0-12)", "teen (13-17)", "adult (18-39)", "middle age (40-59)", "senior (60+)"],
    right=False,
)
titanic["age_group"].value_counts(sort=False, dropna=False)

age_group
child (0-12)           69
teen (13-17)           44
adult (18-39)         438
middle age (40-59)    137
senior (60+)           26
NaN                   177
Name: count, dtype: int64

In [39]:
# 🕵️ Q6 (part 2): Did "women and children first" depend on class?
# observed=True reports only age groups that actually occur.
titanic.pivot_table(values="Survived", index="age_group", columns="Pclass", aggfunc="mean", observed=True).round(2)

Pclass,1,2,3
age_group,,,
child (0-12),0.75,1.00,0.42
teen (13-17),1.00,0.67,0.30
adult (18-39),0.74,0.41,0.23
middle age (40-59),0.58,0.45,0.05
senior (60+),0.29,0.25,0.20


In [40]:
# Always check how many people stand behind each rate.
titanic.pivot_table(values="Survived", index="age_group", columns="Pclass", aggfunc="count", observed=True)

Pclass,1,2,3
age_group,,,
child (0-12),4,17,48
teen (13-17),8,6,30
adult (18-39),93,113,232
middle age (40-59),64,33,40
senior (60+),17,4,5


Every second-class child in our file survived, but only 42% of third-class children did.
Third-class passengers of every age fared worse than passengers of the same age in higher classes.
The first-class child rate (75%) rests on only four children, so treat it with caution.

In [41]:
# qcut() makes four groups of (roughly) equal size: fare quartiles.
titanic["fare_quartile"] = pd.qcut(
    titanic["fare_per_person"], q=4, labels=["lowest 25%", "second 25%", "third 25%", "top 25%"]
)
titanic.groupby("fare_quartile", observed=True)["Survived"].mean().round(3)

fare_quartile
lowest 25%    0.247
second 25%    0.256
third 25%     0.387
top 25%       0.646
Name: Survived, dtype: float64

In [42]:
# 🕵️ Q9: Did travelling with family help?
titanic["family_group"] = pd.cut(
    titanic["family_size"], bins=[0, 1, 4, 11], labels=["alone", "small (2-4)", "large (5+)"]
)

titanic.groupby("family_group", observed=True).agg(
    passengers=("Survived", "size"),
    survival_rate=("Survived", "mean"),
).round(3)

,passengers,survival_rate
family_group,,
alone,537,0.304
small (2-4),292,0.579
large (5+),62,0.161


Passengers in small family groups survived at nearly twice the rate of those travelling alone (57.9% versus 30.4%), but large families fared worst of all (16.1%).
Many large families travelled in third class, so class may explain part of this pattern; that is a good exercise question.

In [43]:
# 🕵️ Q7 (part 2): Was Cherbourg special, or did it carry different passengers?
# normalize="index" turns each row into shares that sum to 1.
pd.crosstab(with_ports["port"], with_ports["Pclass"], normalize="index").round(3)

Pclass,1,2,3
port,,,
Cherbourg,0.506,0.101,0.393
Queenstown,0.026,0.039,0.935
Southampton,0.197,0.255,0.548


In [44]:
# Compare ports WITHIN each class.
with_ports.pivot_table(values="Survived", index="port", columns="Pclass", aggfunc="mean").round(3)

Pclass,1,2,3
port,,,
Cherbourg,0.694,0.529,0.379
Queenstown,0.500,0.667,0.375
Southampton,0.583,0.463,0.190


Half of the Cherbourg passengers travelled first class, compared with about one in five from Southampton, and Cherbourg also had a higher share of women.
So much of the Cherbourg "advantage" reflects *who* boarded there rather than the port itself.
Within each class, Cherbourg passengers still did somewhat better, so class does not explain everything; the remaining gap could reflect other differences we have not measured, and some of these groups are small.
This is **confounding**: a third variable (here, class) is related to both the grouping (port) and the outcome (survival).

In [45]:
# pivot_table() aggregates duplicates, which pivot() cannot do.
# Reshape the Board of Trade sheet (as in the Basic notebook) and total it by class and outcome.
board_tidy = board_of_trade.melt(id_vars="Class", var_name="group", value_name="people")
board_tidy[["age_group", "sex", "outcome"]] = board_tidy["group"].str.split("_", expand=True)

everyone = board_tidy.pivot_table(values="people", index="Class", columns="outcome", aggfunc="sum")
everyone["survival_rate"] = (everyone["Survived"] / (everyone["Survived"] + everyone["Died"])).round(3)
everyone

outcome,Died,Survived,survival_rate
Class,,,
1st,122,203,0.625
2nd,167,118,0.414
3rd,528,178,0.252
Crew,673,212,0.240


> 💭 **Quick thought question:** The first-class child survival rate is based on four children.
How would you report that number to the inquiry so that nobody over-interprets it?

### 🧩 Guided practice: Crosstab practice

Use `pd.crosstab()` to count passengers by `Sex` (rows) and `Pclass` (columns) with `margins=True`.
Then repeat with `normalize="columns"` to show the share of women and men within each class.
Which class had the highest share of women?

In [46]:
# TODO: Counts with margins, then shares within each class.

## 9. Method chaining: a readable pipeline 🔗

Because most pandas methods return a new DataFrame, you can chain them inside parentheses so that the code reads top to bottom like a recipe.
`assign()` adds columns in the middle of a chain; `lambda df: ...` refers to the DataFrame at that point in the chain.
When something goes wrong, comment out the later lines and inspect the intermediate result.

### 🧭 Syntax first: Chaining methods

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
result = (
    df
    .merge(lookup, how="left", on="key")
    .assign(new_column=lambda d: d["a"] / d["b"])
    .groupby("group", as_index=False)
    .agg(total=("value", "sum"))
    .sort_values("total", ascending=False)
)
```

Documentation: [assign](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html) · [pipe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html)

In [47]:
port_report = (
    titanic
    .merge(ports[["code", "port", "stop_number"]], how="left", left_on="Embarked", right_on="code")
    .assign(is_first_class=lambda d: d["Pclass"] == 1,
            is_female=lambda d: d["Sex"] == "female")
    .groupby(["stop_number", "port"], as_index=False)
    .agg(
        passengers=("PassengerId", "size"),
        share_first_class=("is_first_class", "mean"),
        share_female=("is_female", "mean"),
        survival_rate=("Survived", "mean"),
        median_fare_per_person=("fare_per_person", "median"),
    )
    .sort_values("stop_number")
    .round(3)
)
port_report

,stop_number,port,passengers,share_first_class,share_female,survival_rate,median_fare_per_person
0,1.0,Southampton,644,0.197,0.315,0.337,8.90
1,2.0,Cherbourg,168,0.506,0.435,0.554,25.58
2,3.0,Queenstown,77,0.026,0.468,0.390,7.75


In one readable block, the pipeline attaches port names, flags first-class and female passengers, summarizes each port, and orders the ports by the ship's route.

## 10. Bringing it together: the inquiry's report 📝

**Q6 — Class, sex, and age.**
Survival depended on sex first and class second: 96.8% of first-class women survived, compared with 13.5% of third-class men.
"Women and children first" was applied unevenly; every second-class child in our file survived, but only 42% of third-class children did.

**Q7 — Port.**
Cherbourg passengers survived more often (55.4%) mainly because half of them travelled first class and more of them were women; within each class, the port gap shrinks but does not vanish.

**Q8 — Fares.**
Fares belong to tickets, and many tickets covered several people.
Dividing by the number of people on each ticket lowers the median first-class fare from £60.29 to at most £35.25 per person.

**Q9 — Family.**
Small family groups survived more often than people travelling alone, while large families, mostly in third class, fared worst.

**Limitations.**
These patterns describe 891 of roughly 1,300 passengers and no crew; several groups are small; 177 ages are missing; and grouped comparisons show associations, not causes.

## Comprehensive exercises 📝

For each exercise, show a compact result and write two or three sentences of interpretation, including at least one limitation.

### Exercise 1: Rebuild the manifest

Split `titanic` into one DataFrame per port of embarkation, plus a fourth DataFrame for the passengers with a missing port.
Stack all four with `concat()` and `keys=` naming the source, and check that you have 891 rows.
Then merge the result with `ports` using `how="left"`, `validate="many_to_one"`, and `indicator=True`.
Report how many rows matched and list the rows that did not.

In [48]:
# Write your solution here.

### Exercise 2: A class report for the inquiry

Use named aggregation to build one row per `Pclass` with: number of passengers, survivors, survival rate, median age, median `fare_per_person`, share travelling alone, and share with a recorded cabin.
Use `join()` with the `class_names` table so that the rows say "First", "Second", and "Third".
Sort the table by survival rate and write a three-sentence summary.

In [49]:
# Write your solution here.

### Exercise 3: Women and children first, revisited

Build a pivot table of survival rates with `Sex` and `age_group` as the rows and `Pclass` as the columns, plus a second pivot table with the counts behind each rate.
Which group gained the most from the "women and children first" custom, and which group was left behind?
Point to the specific cells that support your answer and flag any rate based on fewer than ten people.

In [50]:
# Write your solution here.

### Exercise 4: Family size or class?

Large families fared worst in Section 8.
Is that because they were large, or because most of them travelled in third class?
Use `crosstab()` and a pivot table of survival by `family_group` and `Pclass` to decide.
Explain your conclusion using the idea of confounding.

In [51]:
# Write your solution here.

### Exercise 5: A reusable report function

Write a function `survival_report(df, by, min_size=1)` that returns, for any grouping column or list of columns `by`, the number of passengers, number of survivors, and survival rate, sorted from the highest rate to the lowest.
Groups with fewer than `min_size` passengers should be dropped (*hint:* `filter()` or a Boolean mask on the result).
Test it with `"Sex"`, `["Pclass", "Sex"]`, `"family_group"`, and `"fare_quartile"`, using `min_size=10` at least once.

In [52]:
# Write your solution here.


def survival_report(df, by, min_size=1):
    pass

## Summary ✅

- `pd.concat()` stacks tables with the same columns (`axis=0`) or places tables side by side by index (`axis=1`).
- `merge()` matches rows on key columns; choose `how=` deliberately, and use `validate=` and `indicator=True` to catch surprises. `join()` is a shortcut for matching against an index.
- `groupby()` follows split-apply-combine. `size()` counts rows, `count()` counts non-missing values, and named aggregation produces readable summary tables.
- `transform()` returns one value per row for within-group comparisons; `filter()` keeps or drops whole groups.
- `pivot_table()`, `crosstab()`, `cut()`, and `qcut()` turn raw rows into comparison tables. Always check group sizes, and look for confounding before drawing conclusions.

**Next:** *Data Manipulation, Cleaning and Imputation* deals with the missing ages, cabins, and ports we kept running into, along with outliers and type conversions.